In [ ]:
import cartopy.crs as ccrs
import numpy as np
import xarray as xr
import pyremo as pr
import cmaps
import glob
from REMO2020_mod import modelclass, season_mean, open_mfdataset, domainclass, plotclass, remo_plotter
from REMO2020TEB_mod import eobs_wind_open

In [ ]:
from dask.distributed import Client, progress
import dask
dask.config.set({'temporary_directory': '/scratch/g/g300096/dask/tmp'})
client = Client()
client

In [ ]:
# define the runs to be plotted
modelruns =[] # init

modelruns.append(modelclass("036041","REMO2020",True))
modelruns.append(modelclass("036043","REMO2020 TEB",True))
modelruns.append(modelclass("036042","REMO2020 iMOVE",True))
modelruns.append(modelclass("036044","REMO2020 TEB iMOVE",True))

In [ ]:
# give path, prefix and suffix (the main name for the plots)
figpath = './figures/'
prefix = 'eobs_wind_seas_'
suffix = 'REMO2020_main'
# or give absolute figure name (leave otherwise empty (''))
afigname = 'fig5'

In [ ]:
# REMO and EOBS datapath
rdata = './data_in/model_data/'
edata = './data_in/measurement_data/E-OBS/'

In [ ]:
# define star year and end year
ystart=2011
yend=2020

In [ ]:
# Define the order of seasons to be used (also when plotting)
seasons = ["DJF","MAM","JJA","SON"]

In [ ]:
# Load the lake mask
lake_mask = xr.open_dataset(rdata+"FLake_input/flake_v3_glcc_defD10.0m_frac_EUR-11.nc")["FLFRA"]

In [ ]:
# lake fraction cut-off limit
clake = 0.5

In [ ]:
# E-OBS version
eobv='v33.0e'

In [ ]:
# set the minimum number days with data per month to be included
dlim=21

In [ ]:
# lake fraction cut-off limit
clake = 0.5

In [ ]:
# Target grid
trggrid = "EUR-11"

In [ ]:
# Eobs variable
var = "fg" # name of the variable to be read in
eobsfile = edata+eobv+"/day/var/"+var+"/"+var+"_ens_mean_0.1deg_reg_"+eobv+".nc"
eobs_mdata, final_mask = eobs_wind_open(var,eobsfile,ystart,yend,trggrid,dlim,lake_mask,clake)

In [ ]:
yearlist = np.arange(ystart,yend+1)
for obj in modelruns:
    obj.filenames = []
    for year in yearlist:
        pattern = rdata+obj.userexp+"/"+str(year)+"/e"+obj.userexp+"m*.nc"
        obj.filenames += glob.glob(pattern)
    obj.filenames.sort()

In [ ]:
# Set halo zone length (will be cutted out)
halo = 8

In [ ]:
# calculate the wind difference to EOBS
ii = 0
for obj in modelruns:
    ds = open_mfdataset(obj.filenames, parallel=False, chunks='auto')
    wim = pr.parse_dates(ds.WIND10) # open data
#    t2m = t2m.squeeze(dim=t2m.dims[1],drop=True)
    # Merge data (get identical coordinates)
    wim_all = xr.merge([wim, final_mask.rename("mask"), eobs_mdata.rename("eobs_m_data")], compat="override", join="override")
    # Calculate absolute values
    if(ii == 0):
        abs_vals = wim_all[["eobs_m_data"]].where(wim_all.mask, drop=False).compute()
        abs_vals_mean = season_mean(abs_vals).eobs_m_data
        abs_vals_mean = abs_vals_mean.where(abs_vals_mean > 0.0, drop=False) # remove zeros
        abs_vals_mean = abs_vals_mean.isel(rlat=slice(halo,abs_vals_mean.sizes["rlat"]-halo), rlon=slice(halo,abs_vals_mean.sizes["rlon"]-halo)).compute()
    # get identical coordinates
    final_tmp = xr.merge([wim_all,abs_vals.eobs_m_data.rename("abs_vals")], compat="override", join="override")
    # apply masking (eobs+lake) and subtract the values, finally making seasonal means
    obj.vardiff = season_mean(final_tmp.WIND10.where(final_tmp.mask,drop=False) - final_tmp.abs_vals).squeeze()
    obj.vardiff = obj.vardiff.isel(rlat=slice(halo,obj.vardiff.sizes["rlat"]-halo), rlon=slice(halo,obj.vardiff.sizes["rlon"]-halo)).compute()
    # not supported, set to zero
    obj.mask = np.zeros(len(seasons))
    # Increase counter
    ii = ii + 1
    #t_stat
    if(obj.r2020file):
        pole = (ds.rotated_latitude_longitude.grid_north_pole_longitude,ds.rotated_latitude_longitude.grid_north_pole_latitude)
        rotated_pole = ccrs.RotatedPole(pole[0], pole[1])
        extent = ([ds.rlon.min(), ds.rlon.max(), ds.rlat.min(), ds.rlat.max()], rotated_pole)
    # clean memory
    del ds, wim, wim_all, final_tmp
# clean memory
del eobs_mdata, abs_vals

In [ ]:
# define the domains to be plotted
domaininfo =[] # init
domaininfo.append(domainclass("EU",[-22.5,47.5,30,72.5], hspace=0.35))

In [ ]:
# define plotclasses for absolute and difference (model) plots
absplotc = plotclass(label='10-m wind speed [m/s]', lspace=np.linspace(0,1,32), vmin=0, vmax=5, linvert=False, cmapp=cmaps.BlAqGrYeOrReVi200)
modplotc = plotclass(label='$\Delta$10-m wind speed [m/s]', lspace=np.linspace(0,1,16), vmin=-2, vmax=2, linvert=False, cmapp=cmaps.BlueWhiteOrangeRed)

In [ ]:
# Loop over domains
for dom in range(len(domaininfo)):
    # Create figure name
    figname=prefix+domaininfo[dom].name+'_'+suffix
    # check if we want to use absolute figure name
    if(afigname):
        figname = afigname
    # Call plotter
    remo_plotter(absplotc,modplotc,domaininfo[dom],modelruns,"E-OBS",abs_vals_mean,seasons,pole,False,figpath,figname)